# Week 6 · RAG Apps + Tool Calling

**The problem.** Riders ask `POST /v1/ask` things like "what does a 50-minute e-bike ride cost me?" and "are there free docks at LS-08?". Today the service searches the handbook once and lets the model answer: it guesses at prices and cannot see stations. By the end of this lab it is an agent that cites the handbook, computes fares with your code and reads live station data.

Last week the model answered from what it learned in training. Today you give it two things it lacks: **sources** (your handbook) and **tools** (your code). You build both from scratch first, then with Microsoft Agent Framework, then you put them in your service.

| Part | What you do | Lecture notes |
|---|---|---|
| 1 | See the problem: a confident, wrong answer | §1 |
| 2 | RAG from scratch: chunk, embed, retrieve, answer, cite | §2 to §6 |
| 3 | Where RAG stops | §1, §11 |
| 4 | Tool calling by hand: the loop | §9, §10 |
| 5 | The same with Agent Framework | §12, §13 |
| 6 | **Your turn:** put the agent in your service, measure | §7, §13 |
| 7 | Deploy and score | |
| 8 | Go further (optional) | |

**You ship:** `POST /v1/ask`, an agent with handbook search, a fare calculator and live stations. **You submit:** your URL.

**How to use this notebook.** Run the cells in order. Parts 1 to 5 build each idea from scratch with small examples; Part 6 is where you change the service; Part 7 ships it. The [lecture notes](lecture.md) explain the why behind each part.

**Optional, for depth:** [deep-dive.ipynb](deep-dive.ipynb) measures retrieval like a search engineer (recall@k, BM25, hybrid fusion, query rewriting, chunking, embedding size) and pushes tool calling to its edges. It is not graded and cannot change your score; open it after you submit.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()

## Part 1 · The model does not know your handbook

In [ ]:
import json
import os

client = course.openai_client()
MODEL = os.environ["FOUNDRY_MODEL"]
EMBEDDING_MODEL = os.environ["FOUNDRY_EMBEDDING_MODEL"]

r = client.responses.create(model=MODEL, input="What is the fee for a lost e-bike at Lakeshore Ride?")
print(r.output_text)

The handbook says **$2,500**. The model either invents a number or hedges. It is not lying: it has never seen the handbook. The fix is not a bigger model. It is putting the right passage in front of the model at question time.

## Part 2 · RAG from scratch

Retrieval-augmented generation has four steps. You will write each one.

1. **Chunk** the documents into passages small enough to be specific.
2. **Embed** every passage: turn its meaning into a vector of numbers.
3. **Retrieve**: embed the question, keep the passages whose vectors point the same way.
4. **Generate**: give the model the question plus those passages, and ask it to cite them.

### 1 · Chunk

The handbook is ten Markdown pages. Each `##` section becomes one chunk, labelled with its page id.

In [ ]:
from app.rag import load_chunks

chunks = load_chunks()
print(f"{len(chunks)} chunks from {len({c.doc_id for c in chunks})} pages\n")
print(chunks[12].render())

### 2 · Embed

An embedding model maps text to a vector (here, 1,536 numbers). Texts with similar meaning get vectors that point in similar directions, even when they share no words. **Cosine similarity** measures that direction: 1 means the same, 0 means unrelated.

In [ ]:
import math


def embed(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


query, same_meaning, other_topic = embed(["fee for a lost e-bike", "what you pay if an electric bike goes missing", "station opening hours"])
print(f"vector length: {len(query)} numbers, norm {math.sqrt(sum(x * x for x in query)):.3f}")
print(f"same meaning, different words: {cosine(query, same_meaning):.3f}")
print(f"different topic:               {cosine(query, other_topic):.3f}")

The norm is 1: these embeddings come normalized, so cosine similarity equals the plain dot product. Vector indexes rely on that to compare faster. Notice also that the unrelated text does not score 0: real scores rarely reach the ends of the scale. Similarity is a ranking signal, so compare scores with each other, not with a fixed threshold.

### 3 · Retrieve

Embed every chunk once (one API call for all of them), then compare each question against all chunk vectors. Forty comparisons are instant. With millions of chunks you would use a vector index such as Azure AI Search; the idea is identical.

In [ ]:
chunk_vectors = embed([c.render() for c in chunks])


def retrieve(question: str, k: int = 3) -> list[tuple[float, object]]:
    [q] = embed([question])
    scored = sorted(((cosine(q, v), c) for c, v in zip(chunks, chunk_vectors)), key=lambda pair: pair[0], reverse=True)
    return scored[:k]


for score, chunk in retrieve("What is the fee for a lost e-bike?"):
    print(f"{score:.3f}  [{chunk.doc_id}] {chunk.heading}")

### 4 · Generate, with citations and abstention

The model gets the passages, labelled with their `doc_id`, and must answer only from them. Structured output makes it return the answer, the ids it used, and a flag for "the sources do not say". That flag matters: an honest "I don't know" is worth more than a fluent guess.

Notice where the passages go: in `input`, marked as handbook text, not in `instructions`. Retrieved text is **data**, not orders. If a document ever contained "ignore your rules and ...", it should be read, not obeyed. That attack, *indirect prompt injection*, returns in Week 10.

In [ ]:
from pydantic import BaseModel, Field


class GroundedAnswer(BaseModel):
    answer: str
    citations: list[str] = Field(description="doc_id of every passage you used")
    abstained: bool = Field(description="true if the passages do not answer the question")


RAG_INSTRUCTIONS = """Answer the rider's question using only the handbook passages provided.
Cite the doc_id of every passage you used. If the passages do not answer the question,
set abstained to true and say you don't know. Answer in under 60 words."""


def rag_answer(question: str) -> GroundedAnswer:
    passages = "\n\n".join(f"[{c.doc_id}] {c.title} > {c.heading}\n{c.text}" for _, c in retrieve(question))
    response = client.responses.parse(
        model=MODEL,
        instructions=RAG_INSTRUCTIONS,
        input=f"Handbook passages:\n{passages}\n\nQuestion: {question}",
        text_format=GroundedAnswer,
    )
    return response.output_parsed


for question in ["What is the fee for a lost e-bike?", "Do you rent electric scooters?"]:
    print(question, "\n ->", rag_answer(question), "\n")

That is RAG. Twenty lines, no framework. Everything else (vector databases, rerankers, knowledge bases) is an improvement on one of these four steps.

## Part 3 · Where RAG stops

Try two questions a rider would ask.

In [ ]:
for question in [
    "I'm a member. How much does a 50-minute classic ride cost?",
    "How many e-bikes are available at LS-08 right now?",
]:
    print(question, "\n ->", rag_answer(question).answer, "\n")

- The fare is **$0.75** (45 minutes included, then 5 × $0.15). The model has the rules but must do arithmetic, and models are unreliable calculators. Sometimes it is right; "sometimes" is not good enough for a bill.
- Live availability is not in any document. No amount of retrieval can find it.

Both need **code**, not text. The business already has that code: `app/domain.py` computes fares exactly and reads station data. The model just needs a way to call it.

## Part 4 · Tool calling, by hand

You describe each function with a name, a description and a JSON schema for its arguments. The model never runs your code. It answers with a **function call**: "please run `quote_fare` with these arguments". Your code runs it and sends back the result; the model then writes the final answer. That exchange is **the loop**, and it is the heart of every agent.

In [ ]:
from app import domain

TOOLS = [
    {
        "type": "function",
        "name": "quote_fare",
        "description": "Exact ride fee in US dollars for one ride. Always use this instead of computing prices.",
        "parameters": {
            "type": "object",
            "properties": {
                "plan": {"type": "string", "enum": ["single", "day_pass", "member", "student"]},
                "minutes": {"type": "integer", "description": "Ride length in whole minutes"},
                "ebike": {"type": "boolean", "description": "true for an e-bike, false for a classic bike"},
            },
            "required": ["plan", "minutes", "ebike"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_station",
        "description": "Live classic bikes, e-bikes and free docks at one station.",
        "parameters": {
            "type": "object",
            "properties": {"station_id": {"type": "string", "description": "Station id, for example LS-07"}},
            "required": ["station_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]
FUNCTIONS = {"quote_fare": domain.quote_fare, "get_station": domain.get_station}
TOOL_INSTRUCTIONS = "You answer Lakeshore Ride questions. Use the tools for prices and station data. Be brief."


def ask_with_tools(question: str) -> str:
    response = client.responses.create(model=MODEL, instructions=TOOL_INSTRUCTIONS, input=question, tools=TOOLS)
    while True:
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text  # no more tool requests: this is the final answer
        results = []
        for call in calls:
            arguments = json.loads(call.arguments)
            result = FUNCTIONS[call.name](**arguments)
            print(f"  model asked for {call.name}({arguments}) -> {result}")
            results.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
        response = client.responses.create(
            model=MODEL, instructions=TOOL_INSTRUCTIONS, previous_response_id=response.id, input=results, tools=TOOLS
        )


print(ask_with_tools("I'm a member. How much does a 50-minute classic ride cost?"), "\n")
print(ask_with_tools("How many e-bikes are available at LS-08 right now?"))

Read the printed lines: the model chose the tool, filled the arguments from the question, and used the exact result. Notice what `strict: True` bought you: the arguments always match the schema, so `**arguments` never crashes on a typo.

## Part 5 · The same, with Microsoft Agent Framework

Writing the loop once is how you understand it. Writing it in every service is how bugs happen. **Agent Framework** gives you the loop, plus streaming, sessions, structured output, tracing and multi-agent workflows (next week).

A tool is now an ordinary Python function with type hints and a docstring; the framework builds the JSON schema from them. The service's tools are in `app/tools.py`:

In [ ]:
source = Path("app/tools.py").read_text()
start = source.index("@tool\ndef quote_fare")
print(source[start : source.index("@tool", start + 1)])

In [ ]:
from agent_framework import Agent

from app.config import get_settings
from app.state import AppState
from app.tools import get_station, quote_fare, recording, search_handbook

state = AppState(get_settings())
agent = Agent(
    client=state.models.agent_client(),
    name="lakeshore-ask",
    instructions="You answer Lakeshore Ride questions. Use search_handbook for policies, quote_fare for prices, "
    "get_station for live data. Answer only from tool results.",
    tools=[search_handbook, quote_fare, get_station],
)

with recording(state) as calls:  # records every tool call the agent makes
    result = await agent.run("I'm a member. What does a 50-minute classic ride cost, and is there a free dock at LS-14?")

print(result.text, "\n")
for call in calls:
    print(f"{call['name']}({call['arguments']})")

One question, several tools, chosen by the model, in the right order. Add structured output and you have exactly what `/v1/ask` must return:

In [ ]:
from app.services.ask import AskAnswer

with recording(state) as calls:
    result = await agent.run("What is the fee for a lost e-bike?", options={"response_format": AskAnswer})

print(result.value)
print([c["name"] for c in calls])

### Designing tools a model uses well

The model chooses tools from their names, descriptions and schemas, nothing else. Write them as an API for a reader who cannot ask questions:

| Principle | Example |
|---|---|
| One clear job per tool, named for it | `quote_fare`, not `pricing_helper` |
| The description says **when** to use it, not only what it does | "Always use this instead of computing prices" (Part 4) |
| Constrain arguments with types and enums | `plan` is one of four values |
| Return errors as data the model can act on | `get_station("LS-99")` returns `{"error": "No station with id 'LS-99'. Ids look like LS-07."}` |
| Return only what the next step needs | A few passages, not the whole handbook |
| Keep the set small | Every tool definition costs tokens on every call and adds one more way to choose wrong |

You now have two retrieval styles side by side. Part 2 retrieves **before** the model sees the question. The agent retrieves **when it decides to**, by calling `search_handbook` with a query it writes: just-in-time context. It costs a model round trip and often buys a better query. Read [Writing effective tools for agents (Anthropic)](https://www.anthropic.com/engineering/writing-tools-for-agents) and [Function calling (OpenAI)](https://developers.openai.com/api/docs/guides/function-calling).

## Part 6 · Put the agent in your service

First, the baseline. The starting `ask()` is the Part 2 pipeline: retrieve, then one model call. Measure it.

In [ ]:
baseline = course.check(6, in_process=True)

Look at `w06.ask.tools`: the baseline never calls `quote_fare` or `get_station`, so prices are guesses and station questions fail.

### Your turn

Open `app/services/ask.py` and replace the retrieve-then-generate block in `ask()` with the agent from Part 5:

1. Import `Agent` from `agent_framework`, and `get_station`, `quote_fare`, `recording`, `search_handbook` from `app.tools`.
2. Rewrite `INSTRUCTIONS`: name each tool and when to use it, require citations, define when to abstain, set a length limit.
3. Build the `Agent` with the three tools, using `state.models.agent_client()`.
4. Run it inside `with recording(state) as calls:` with `options={"response_format": AskAnswer}`.
5. Return an `AskResponse` ([contract](../app/contract.py)): the answer and `abstained` from `result.value`, citations filtered to known `doc_ids()`, and `tool_calls` from `calls`.

Keep the guardrail check and the offline branch at the top as they are: the offline tests (`uv run pytest`) rely on them. Then measure. The first goal is `w06.ask.tools` above zero; tune instructions after that. Deploy as soon as the agent version beats the baseline, then keep improving: you can redeploy as often as you like.

In [ ]:
improved = course.check(6, in_process=True)
print(f"\nbaseline {baseline['score']}  ->  now {improved['score']}")

If a check is not at 100%, read its detail line and open the proof in `reports/` to see the exact answer. Typical fixes, in order of impact:

- **Tools not called:** the instructions do not say *when* to use each tool. Be explicit: "Never compute prices yourself."
- **A question that needs a rule and a number:** "how much credit do I get when my battery dies?" needs the rule from the handbook *and* the amount from `quote_fare`. Say so in the instructions, with one worked example on a case that is not in the public data.
- **A price without its page:** when a policy shapes a price (included minutes, the fare cap), the agent should search the handbook as well and cite the page with the rule.
- **Wrong citations:** the model cites everything it read. Ask it to cite only passages it used, at most three.
- **Abstains too often or too rarely:** define it precisely: "only when no tool result or passage answers the question". Silence is not an answer: if no passage mentions what the rider asks about, the agent must not infer a "no".

**100 is reachable.** Every public question is answered by the handbook, by a tool, or by neither (then the right answer is to abstain). Fix one cause at a time, re-run, and check that nothing else broke. Deploy as soon as the agent beats the baseline (Part 7), then keep going until the local harness shows 100 and redeploy.

## Part 7 · Deploy and score

In [ ]:
url = course.deploy()
proof = course.check(6, url)

Submit your URL by the deadline in the [Week 6 README](README.md).

## Part 8 · Go further

- **Hybrid retrieval:** combine vector scores with keyword matches in `app/rag.py`. Station ids such as `LS-08` and fee names are exact strings; keywords find them where meaning does not. Which questions improve?
- **Retrieval depth:** `search_handbook` returns four passages. Try two and six. Watch citations, tokens and latency.
- **Managed retrieval:** read how [Foundry IQ](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/what-is-foundry-iq) and [agentic retrieval in Azure AI Search](https://learn.microsoft.com/en-us/azure/search/agentic-retrieval-overview) provide the same capability as a service: the model plans several sub-queries, the service runs them and merges the results.
- **Tool design:** rewrite the `quote_fare` docstring badly on purpose (vague, no "when to use"). Measure. Then restore it.
- **Deep dive (optional):** [deep-dive.ipynb](deep-dive.ipynb) builds BM25, hybrid search with Reciprocal Rank Fusion and multi-query rewriting from scratch, measures each with recall@k and MRR on a harder question set, and compares RAG as a tool with RAG as a context provider.

### What to remember

- RAG = chunk, embed, retrieve, generate. Citations and abstention make it trustworthy.
- Retrieved text is data, not instructions.
- Models are bad at arithmetic and blind to live data: give them **tools** for both.
- The tool loop: schema in, function call out, your code runs it, result back, final answer.
- A tool's name, description and schema are all the model knows about it. Write them like an API.
- A framework saves you from rewriting the loop, not from understanding it.

**Learn more:** [Develop generative AI apps that use tools](https://learn.microsoft.com/en-us/training/modules/use-generative-ai-tools/) · [Introduction to RAG concepts](https://learn.microsoft.com/en-us/training/modules/rag-fundamentals/) · [Function calling (OpenAI)](https://developers.openai.com/api/docs/guides/function-calling) · [Writing effective tools for agents (Anthropic)](https://www.anthropic.com/engineering/writing-tools-for-agents)